# RETFound Step 5 on Google Colab

This notebook continues the existing Step 5 pipeline for RETFound without changing the ConvNeXt code or outputs. Large datasets and `.pth` checkpoints remain in Google Drive; only the resulting small `retfound_config.json` score summary is prepared for GitHub.

In [ ]:
# Edit these paths if your Google Drive folders differ.
REPOSITORY_URL = "https://github.com/ZumerDhillun/DR-Screening-Study.git"
BRANCH = "main"
DRIVE_ARCHIVES = "/content/drive/MyDrive/DR-Screening-Study-data/archives"
RETFOUND_CHECKPOINT = "/content/drive/MyDrive/retfound/RETFound_mae_natureCFP.pth"
DRIVE_OUTPUT = "/content/drive/MyDrive/retinavision_models"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then reconnect."
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/DR-Screening-Study")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
%cd /content/DR-Screening-Study
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## Extract the raw datasets onto Colab's local disk

Google Drive should contain `ddr.zip`, `deepdrid.zip`, and `aptos2019.zip` below `DRIVE_ARCHIVES`. Extracting once per Colab session is much faster than training through thousands of Drive-mounted image files. Step 2 searches recursively, so extra wrapper folders inside an archive are acceptable.

In [ ]:
import shutil

archive_root = Path(DRIVE_ARCHIVES)
raw_root = Path("data/raw")
raw_root.mkdir(parents=True, exist_ok=True)
for dataset in ("ddr", "deepdrid", "aptos2019"):
    archive = archive_root / f"{dataset}.zip"
    destination = raw_root / dataset
    assert archive.is_file(), f"Missing Drive archive: {archive}"
    if destination.exists() and any(destination.rglob("*")):
        print(f"Reusing extracted dataset: {destination}")
    else:
        destination.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {archive.name} ...")
        shutil.unpack_archive(str(archive), str(destination))
    file_count = sum(path.is_file() for path in destination.rglob("*"))
    print(f"{dataset}: {file_count:,} extracted files")

checkpoint = Path(RETFOUND_CHECKPOINT)
assert checkpoint.is_file(), f"Checkpoint not found: {checkpoint}"
print(f"Checkpoint ready: {checkpoint.name} ({checkpoint.stat().st_size / 1e9:.2f} GB)")

## Rebuild Colab-native paths and verify preprocessing

The committed CSV paths may use Windows separators. Re-running Steps 2 and 3 in Colab produces Linux paths while retaining the partner pipeline's grade mapping and seeded split.

In [ ]:
subprocess.run(["python", "scripts/step2_binarize_labels.py"], check=True)
subprocess.run(["python", "scripts/step3_class_imbalance_split.py"], check=True)
subprocess.run(["python", "scripts/step4_preprocessing_pipeline.py"], check=True)

## Train RETFound

This invokes the additive RETFound launcher, which delegates to the unchanged `step5_train.py`. Re-running resumes from the Drive checkpoint written after each epoch.

In [ ]:
subprocess.run([
    "python", "scripts/step5_retfound.py",
    "--checkpoint", RETFOUND_CHECKPOINT,
    "--epochs", "15",
    "--batch_size", "8",
    "--lr", "1e-4",
    "--out_dir", DRIVE_OUTPUT,
], check=True)

## Prepare the score summary for GitHub

The JSON contains Step 5 validation AUROC, temperature, decision threshold, and class weight. It does **not** contain model weights or patient images.

In [ ]:
import json
import shutil

drive_result = Path(DRIVE_OUTPUT) / "retfound_config.json"
assert drive_result.is_file(), f"Training result not found: {drive_result}"
github_result = Path("configs/retfound_config.json")
shutil.copy2(drive_result, github_result)
print(json.dumps(json.loads(github_result.read_text()), indent=2))
subprocess.run(["git", "status", "--short"], check=True)

# Also download the small result file so it can be committed from your local clone.
from google.colab import files
files.download(str(github_result))

## Submit the result

Review `git status`. Commit only the notebook, launcher, and `configs/retfound_config.json`; never add `.pth` files or `data/raw`. In a terminal authenticated with GitHub, run:

```bash
git add notebooks/retfound_colab.ipynb scripts/step5_retfound.py configs/retfound_config.json
git commit -m "Add RETFound Colab training results"
git push origin main
```

The trained `retfound_best.pth` and resumable `retfound_checkpoint.pth` remain safely in Google Drive.